# 직접 수집 데이터 둘러보기

첫날 수집이 끝나면 `python -m src.preprocess`로 `D:/datasets/processed/self_collected.npz`를 만든 뒤 위에서부터 실행한다.

여기서 정할 것:

1. **`--skip-first` 값** — 인스타 화면 분류 파일에서 캡처를 켠 직후 몇 초 동안 연결이 몰리는지
2. **TCP(TLS)도 쓸지, QUIC만 쓸지** (`--proto`) — 라벨마다 TCP·QUIC 비율
3. **라벨별 플로우 수** — 너무 치우치면 macro-F1이 흔들린다

- 커널은 저장소의 `.venv`를 고른다.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # notebooks/ 에서 src 를 불러오기 위해

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import DIR, IPT, PROCESSED_DIR, SIZE, TASKS

plt.rcParams["font.family"] = "Malgun Gothic"  # 한글 글꼴
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
npz = np.load(PROCESSED_DIR / "self_collected.npz")
ppi = npz["ppi"]
flows = pd.DataFrame({
    "label": npz["label"],
    "date": npz["date"],
    "session": npz["session"],
    "proto": pd.Series(npz["proto"]).map({"udp": "QUIC", "tcp": "TLS"}),
    "offset": npz["offset"],  # 캡처 시작부터 플로우가 열릴 때까지 (초)
    "ppi_len": (ppi[:, DIR] != 0).sum(1),  # 앞 30개 중 실제로 채워진 패킷 수
})
LABELS = [label for labels in TASKS.values() for label in labels if label in set(flows["label"])]
print(f"플로우 {len(flows):,}개 · 파일 {flows['session'].nunique()}개 · 날짜 {sorted(flows['date'].unique())}")

## 라벨별 플로우 수

In [ ]:
pd.crosstab(flows["label"], flows["date"], margins=True, margins_name="합계").reindex(LABELS + ["합계"])

In [ ]:
# 파일 하나(5분)에서 나오는 플로우 수
per_file = flows.groupby(["label", "session"]).size().groupby("label")
pd.DataFrame({"파일": per_file.size(), "평균": per_file.mean().round(1), "최소": per_file.min(), "최대": per_file.max()}).reindex(LABELS)

## ① `--skip-first` — 캡처를 켠 직후에 연결이 몰리는지

화면 분류(`screen`)는 인스타를 켜 둔 채 캡처를 켜서, 처음 몇 초에 재연결이 몰릴 수 있다. 그 연결은 그 화면과 상관없을 수 있으니 뺀다.
앱 분류(`app`)와 비교해서 **화면 분류 쪽만 앞부분이 튀어 있으면, 튀는 구간이 끝나는 초**를 `--skip-first`로 쓴다.
(앱 분류는 앱을 켜는 순간 연결이 몰리는 게 정상이다 — 앱 분류에는 쓰지 않는다.)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
bins = np.arange(0, 61, 1)
for ax, (task, labels) in zip(axes, TASKS.items()):
    for label in labels:
        mine = flows[flows["label"] == label]
        if len(mine):  # 파일 수가 달라도 비교할 수 있게 파일당 플로우 수로
            ax.hist(mine["offset"], bins=bins, histtype="step", linewidth=1.5, label=label,
                    weights=np.full(len(mine), 1 / mine["session"].nunique()))
    ax.set_title(f"{task} — 처음 60초")
    ax.set_xlabel("캡처 시작부터 플로우가 열릴 때까지 (초)")
    ax.legend()
axes[0].set_ylabel("파일당 플로우 수 (1초 구간)")
plt.tight_layout()

In [ ]:
# --skip-first 값마다 빠지는 플로우 비율 — 너무 크게 잡으면 데이터가 줄어든다
pd.DataFrame({
    f"{s}초": flows.groupby("label")["offset"].apply(lambda offset: (offset < s).mean()) for s in [3, 5, 10, 15, 20, 30]
}).reindex(LABELS).style.format("{:.0%}")

## ② TCP(TLS)도 쓸지 — 라벨마다 프로토콜 비율

CESNET은 QUIC만이다. 직접 수집에서 어떤 라벨만 거의 TCP를 쓰면, 모델이 패킷 크기 모양으로 「TCP냐 QUIC이냐」를 알아채서
앱 대신 프로토콜로 맞힐 수 있다. 그렇다면 `--proto all`과 `--proto quic` 결과를 둘 다 적는다.

In [ ]:
counts = pd.crosstab(flows["label"], flows["proto"]).reindex(LABELS)
counts.join(counts.div(counts.sum(1), axis=0).add_suffix(" 비율").round(2))

## ③ 플로우 길이 — 앞 30개를 다 채우는지

패킷이 몇 개 안 되는 짧은 플로우는 정보가 적어 맞히기 어렵다. 라벨마다 짧은 플로우 비율이 크게 다르면 그 자체가 지름길이 될 수 있다.

In [ ]:
by_label = flows.groupby("label")["ppi_len"]
pd.DataFrame({
    "중앙값": by_label.median(),
    "30개 다 참": by_label.apply(lambda n: (n == 30).mean()),
    "5개 미만": by_label.apply(lambda n: (n < 5).mean()),
}).reindex(LABELS).style.format({"중앙값": "{:.0f}", "30개 다 참": "{:.0%}", "5개 미만": "{:.0%}"})

## ④ 라벨별 모양 — 앞 30개 패킷

In [ ]:
# 라벨마다 플로우 20개의 (방향 × 크기) — 위쪽은 클라이언트→서버, 아래쪽은 서버→클라이언트. 윗줄 앱 분류, 아랫줄 화면 분류
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True, sharey=True)
for ax, label in zip(axes.flat, [label for labels in TASKS.values() for label in labels]):
    mine = ppi[(flows["label"] == label).to_numpy()]
    for flow in mine[:20]:
        n = int((flow[DIR] != 0).sum())
        ax.plot(flow[DIR, :n] * flow[SIZE, :n], alpha=0.4)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.set_title(f"{label} ({len(mine)})")
for ax in axes[1]:
    ax.set_xlabel("패킷 순서")
for ax in axes[:, 0]:
    ax.set_ylabel("방향 × 크기 (바이트)")
plt.tight_layout()

In [ ]:
# 패킷 간격(ms) 분포 — 첫 패킷(간격 0)은 뺀다
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (task, labels) in zip(axes, TASKS.items()):
    for label in labels:
        mine = ppi[(flows["label"] == label).to_numpy()]
        ipt = mine[:, IPT, 1:][mine[:, DIR, 1:] != 0]
        if len(ipt):
            ax.hist(np.log1p(ipt), bins=60, histtype="step", density=True, label=label)
    ax.set_title(task)
    ax.set_xlabel("log(1 + 간격 ms)")
    ax.legend()
plt.tight_layout()

## 정리

정한 값과 근거를 todo.md에 적고 학습 명령에 쓴다.

- `--skip-first`: ?초 — 근거:
- `--proto`: all / quic — 근거:
- 라벨별 플로우 수가 치우친 라벨: